[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippe-amuzuga/Meca-IA_Ed2026/blob/main/meca-ia/C13/E13-1_Pompe_Similitude.ipynb)

# Lois de similitude d'une pompe : du neurone au réseau

## Le contexte

Une pompe centrifuge élève la pression d'un fluide. La hauteur manométrique qu'elle fournit dépend du débit, de la vitesse de rotation, du diamètre de la roue et du fluide pompé, selon des lois de similitude qu'on sait écrire pour une pompe idéale, mais pas pour une machine réelle, avec ses pertes. Un banc d'essai a relevé 800 points de fonctionnement. Le bureau d'études voudrait apprendre cette loi à partir des mesures, sans l'écrire.

La grandeur à prédire est une hauteur en mètres : c'est un problème de régression.

Défi : prédire `hauteur_m` et obtenir un R² supérieur à celui du modèle de départ sur la base de test. Ce modèle de départ est un neurone isolé, une somme pondérée des entrées entraînée par descente de gradient ; il atteint un R² de 0,75. À vous d'ajouter des couches de neurones : la section « Pour aller plus loin » montre comment la rétropropagation les entraîne. La règle de découpage est fixée plus bas. La base de test sert à mesurer, jamais à apprendre ni à régler.

## Les données

Fichier `MecaIA-Pompe_Similitude.xlsx`, dans le dossier `donnees` du dépôt : 800 mesures, 6 colonnes. Une ligne est un point de fonctionnement mesuré sur un banc d'essai de pompe.

| Colonne | Signification | Unité |
|---|---|---|
| `fluide` | fluide pompé : `eau` ou `huile`, plus visqueuse | catégorie |
| `diametre_roue_mm` | diamètre extérieur de la roue | mm |
| `vitesse_rpm` | vitesse de rotation de l'arbre | tr/min |
| `debit_m3_h` | débit traversant la pompe | m³/h |
| `hauteur_m` | cible : hauteur manométrique mesurée | m |
| `puissance_kW` | puissance absorbée à l'arbre, seconde cible du jeu, écartée ici : elle se calcule à partir de la hauteur | kW |

Origine : jeu simulé, d'après les lois de similitude des pompes centrifuges, avec une dispersion d'une pompe à l'autre et le bruit des capteurs. Quelques vitesses de rotation ne sont pas relevées. Il enseigne une méthode, et aucune conclusion d'hydraulique n'en découle.

### Étape 1 : charger le fichier et regarder sa forme

On lit le classeur, puis on vérifie ses dimensions, ses cases vides et la plage de la cible avant toute modélisation : la forme d'un fichier se lit sur le fichier entier. Le dossier des données est cherché depuis le dossier du notebook, puis depuis un clone du dépôt ; ouvert seul, par exemple dans Colab, le notebook télécharge une fois le jeu depuis le dépôt public.

In [ ]:
from pathlib import Path
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import set_config

set_config(display="text")
GRAINE = 42
FICHIER = "MecaIA-Pompe_Similitude.xlsx"
URL_JEU = "https://raw.githubusercontent.com/philippe-amuzuga/Meca-IA_Ed2026/main/donnees/MecaIA-Pompe_Similitude.xlsx"

# Le dossier `donnees` du dépôt, cherché depuis le dossier du notebook, puis depuis un clone.
candidats = [Path("..", "..", "donnees"), Path("Meca-IA_Ed2026", "donnees")]
dossier = next((d for d in candidats if (d / FICHIER).is_file()), None)
if dossier is None:
    # Notebook ouvert seul (Colab) : copie unique du jeu depuis le dépôt public.
    dossier = Path("donnees")
    dossier.mkdir(exist_ok=True)
    try:
        urllib.request.urlretrieve(URL_JEU, dossier / FICHIER)
    except OSError as erreur:
        raise FileNotFoundError(
            "Jeu introuvable : clonez le dépôt, ou vérifiez la connexion pour le télécharger.") from erreur
df = pd.read_excel(dossier / FICHIER)

cible = "hauteur_m"
categories = ["fluide"]
mesures = ["diametre_roue_mm", "vitesse_rpm", "debit_m3_h"]
print("dimensions :", df.shape)
vides = df.isna().sum()
print("cases vides :", vides[vides > 0].to_dict())
print(df[cible].describe().round(1).to_dict())
df.head(3)

Lecture du résultat : 800 mesures, 6 colonnes. La vitesse de rotation compte 21 cases vides, qu'il faudra remplir. La hauteur va de 1,8 à 149,8 m, pour une médiane de 25,9 m et une moyenne de 32,9 m : quelques points de fonctionnement fournissent des hauteurs très supérieures aux autres.

### Étape 2 : mettre le test de côté

Un quart des mesures est mis de côté avant tout apprentissage, tout réglage et toute comparaison avec la cible. La graine 42 fixe le tirage : chacun obtient la même base de test, et les scores se comparent. La puissance, seconde cible du jeu, ne sert pas d'entrée : elle dépend de la hauteur.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["hauteur_m", "puissance_kW"])
y = df[cible]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=GRAINE)
print("apprentissage :", len(X_train), "mesures")
print("test :", len(X_test), "mesures")

### Étape 3 : regarder la hauteur contre le débit sur l'apprentissage

La figure place la hauteur des mesures d'apprentissage selon le débit, avec un symbole par fluide.

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 3.8))
for fluide, marque, gris in [("eau", "o", "0.75"), ("huile", "^", "0.1")]:
    choix = X_train["fluide"] == fluide
    ax.scatter(X_train.loc[choix, "debit_m3_h"], y_train[choix], s=18, marker=marque, color=gris,
               edgecolor="black", linewidth=0.4, label=fluide)
ax.set(xlabel="débit (m³/h)", ylabel="hauteur manométrique (m)")
ax.legend(frameon=False)
ax.grid(alpha=0.3)
plt.show()
bande = (X_train["debit_m3_h"] > 40) & (X_train["debit_m3_h"] < 60)
print(f"débit entre 40 et 60 m³/h : {bande.sum()} mesures, hauteur de {y_train[bande].min():.1f} à {y_train[bande].max():.1f} m")

Lecture du résultat : à débit voisin (entre 40 et 60 m³/h), la hauteur va de 4,2 à 91,1 m. Le débit seul n'explique donc pas la hauteur : la vitesse de rotation et le diamètre de la roue interviennent aussi, et le nuage n'a pas la forme d'une droite. Un modèle qui additionne les entrées avec un poids chacune ne pourra suivre qu'une partie de cette loi.

### Étape 4 : préparer les entrées

Le fluide devient une colonne de 0 et de 1, les vitesses non relevées sont remplacées par la médiane de l'apprentissage, et chaque entrée est ramenée à une moyenne nulle et à un écart type de 1. Un neurone, comme un réseau, apprend mal quand ses entrées n'ont pas la même échelle. La préparation s'apprend sur `X_train` seul et s'applique ensuite au test.

In [ ]:
# Votre code :

### Étape 5 : construire le modèle de départ, un neurone

Un neurone calcule une somme pondérée de ses entrées, plus un biais : `sortie = w · x + b`. L'apprentissage cherche les poids `w` et le biais `b` qui rendent petite l'erreur quadratique moyenne entre la sortie et la hauteur mesurée. Il procède par descente de gradient : à chaque époque, il calcule l'écart de la sortie à la cible, en déduit la pente de l'erreur par rapport à chaque poids, puis fait un petit pas dans le sens qui la diminue. La hauteur est d'abord réduite (moyenne nulle, écart type 1), pour que le pas ait le même sens quelle que soit l'unité.

In [ ]:
# Votre code :

### Étape 6 : mesurer sur la base de test

Le neurone prédit les mesures qu'il n'a jamais vues. Trois indicateurs sont lus : le coefficient de détermination R², et deux erreurs en mètres, la racine de l'erreur quadratique moyenne (RMSE) et l'erreur absolue moyenne (MAE). Un témoin sans apprentissage, qui répond toujours la moyenne de l'apprentissage, donne le point de comparaison le plus bas.

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error

score_depart = r2_score(y_test, y_pred)
score_temoin = r2_score(y_test, np.full(len(y_test), y_train.mean()))
print(f"R² : {score_depart:.3f}")
print(f"RMSE : {root_mean_squared_error(y_test, y_pred):.2f} m")
print(f"MAE : {mean_absolute_error(y_test, y_pred):.2f} m")
print(f"R² du témoin (toujours la moyenne de l'apprentissage) : {score_temoin:.3f}")
# Votre code :

Lecture du résultat : le R² du neurone est de 0,75 et l'erreur absolue moyenne de 8,3 m, pour une hauteur moyenne de 33,8 m sur la base de test. Le témoin, qui répond toujours la moyenne, n'explique rien : son R² est nul, au signe de l'arrondi près. La régression linéaire résolue en une ligne donne le même R² (0,746 contre 0,746) : un neurone sans courbure est une régression linéaire, et la descente de gradient en retrouve la solution en faisant passer l'erreur quadratique réduite de 0,94 à 0,20. C'est ce chiffre, mesuré sur des mesures que le modèle n'a jamais vues, que l'on a le droit d'annoncer.

## Pour aller plus loin : une couche cachée et la rétropropagation

Le modèle de départ de l'étape 5 reste le score de référence du défi. Les quatre cellules suivantes montrent comment le dépasser sans regarder le test pour choisir. Un neurone ne trace qu'un plan ; une couche de neurones à fonction tanh placée entre les entrées et la sortie lui permet de courber la loi. La rétropropagation calcule la pente de l'erreur couche par couche : l'écart de la sortie à la cible est renvoyé vers la couche cachée, pondéré par les poids de sortie et par la pente de la fonction tanh, puis chaque poids fait son petit pas.

La première cellule écrit l'entraînement. La deuxième compare plusieurs largeurs de la couche cachée sur un cinquième de l'apprentissage mis de côté pour juger (la validation). Les échelles des entrées et de la cible s'apprennent sur les quatre cinquièmes d'ajustement. La troisième cherche, sur l'apprentissage seul et en cinq plis, où le réseau retenu se trompe. La quatrième entraîne la largeur retenue sur tout l'apprentissage et la mesure une seule fois sur le test. Les valeurs sont lues dans les sorties imprimées.

In [ ]:
# Votre code :

In [ ]:
# Votre code :

In [ ]:
# Votre code :

In [ ]:
# Votre code :

Lecture du résultat : la validation retient 16 neurones cachés (R² de validation de 0,993, contre 0,967 pour 2 neurones). Hors pli, sur l'apprentissage seul, le réseau atteint un R² de 0,97, mais les 17 relevés sans vitesse, soit 2,8 % des mesures, portent 63 % de l'erreur quadratique : remplacée par la médiane, la vitesse manque, et la hauteur varie comme son carré. Aucun réseau ne devine une vitesse qu'on ne lui donne pas ; sur une base de 200 mesures, quelques relevés de ce type suffisent à dominer l'erreur. Mesuré une seule fois sur le test, le réseau atteint un R² de 0,88, soit 0,13 de plus que le neurone seul : il apprend la courbure de la loi qu'aucune droite ne suit. La section suivante empile davantage de couches avec le réseau de scikit-learn.

## Le score de référence

Le score publié est le R² du neurone de l'étape 5 sur la base de test. L'empreinte identifie cette base de test : si la vôtre diffère, le découpage n'est pas le même et les scores ne se comparent pas.

In [ ]:
import hashlib

score = r2_score(y_test, y_pred)
positions = ",".join(str(i) for i in sorted(X_test.index))
print("EMPREINTE_TEST =", hashlib.sha256(positions.encode("ascii")).hexdigest())
print("SCORE_REFERENCE r2 = %.2f" % score)

## Synthèse

1. Un neurone calcule une somme pondérée de ses entrées plus un biais ; sans fonction de courbure, c'est une régression linéaire, et la descente de gradient en retrouve la solution (R² de 0,75 contre 0,75 en résolution directe).
2. L'apprentissage mesure un écart par une erreur quadratique moyenne et déplace chaque poids d'un petit pas dans le sens qui la diminue ; les entrées doivent avoir la même échelle pour que ce pas convienne à tous les poids.
3. Une couche cachée à fonction tanh courbe la loi que le neurone seul ne suit pas ; la rétropropagation renvoie l'écart de la sortie vers cette couche pour calculer la pente de chaque poids.
4. La largeur de la couche cachée se règle sur un cinquième de l'apprentissage mis de côté, jamais sur le test (16 neurones retenus ici).
5. Le test ne sert qu'une fois : le réseau retenu y atteint un R² de 0,88, le neurone seul de 0,75.